## Thresholds For Gating & Blending

**Author:** Jakob Balkovec

In [1]:
import pandas as pd
import numpy as np

path = "/Users/jbalkovec/Desktop/MDR/Models/Temporal/BEST_MODEL/test_predictions_weighted.csv"
target_col = "soil_moisture_5cm"


In [2]:
preds = pd.read_csv(path)

if target_col not in preds.columns:
    raise ValueError(f"{target_col} not found in predictions file")

In [3]:
g1, g2 = np.percentile(preds[target_col].values, [33, 66])

print("Deployable thresholds:")
print("g1 (33rd):", g1)
print("g2 (66th):", g2)

Deployable thresholds:
g1 (33rd): 0.179
g2 (66th): 0.271


I treated the thresholds as hyperparameters instead of fixed values and just tuned them on validation. Instead of using the ones from the true target distribution, I ran a simple grid search over different (g1, g2) pairs using the base model predictions as the gating signal. For each pair, I applied soft gating, computed the final predictions, and picked the one that gave the best R² on validation. Then I locked those thresholds in for deployment so they actually match how the model behaves.

**Optimal Thresholds Found:**

`FORMAT: R^2, t1, t2`

`(0.9589674852506406, 0.19090909090909092, 0.2909090909090909)`

In [4]:
print(np.percentile(preds[target_col], [5, 25, 50, 75, 95]))

[0.033 0.136 0.234 0.288 0.328]


In [ ]:
from sklearn.metrics import r2_score

g1_grid = np.linspace(0.14, 0.22, 12)
g2_grid = np.linspace(0.24, 0.32, 12)

best = None

for g1 in g1_grid:
    for g2 in g2_grid:
        if g1 >= g2:
            continue

        w_d, w_t, w_w = soft_weights_from_signal(pred_val_base, g1, g2, blend_width)

        pred_val_tmp = (
            w_d * pred_val_dry_full +
            w_t * pred_val_transition_full +
            w_w * pred_val_wet_full
        )

        keep = np.isfinite(pred_val_tmp)
        if keep.sum() == 0:
            continue

        r2 = r2_score(y_val.iloc[keep], pred_val_tmp[keep])

        if best is None or r2 > best[0]:
            best = (r2, g1, g2)

print("Best (VAL):", best)